# RAG Vector DB Setup — **BAAI/bge-m3** variant (Milestone 3)

> **Model-comparison run.** Identical to `08_rag_vector_db_setup.ipynb` (which uses
> `intfloat/multilingual-e5-base`) except for the embedding model and its prefix
> convention. Same chunks, same chunking, same 512-token truncation, same gate,
> same eval battery, same inspector queries — so the scorecards are comparable.
> Writes to its own `rag_outputs_bge_m3_*` folder; nothing is overwritten.

Builds the production retrieval stack specified in the Milestone-3 report (Sections 9-10):
one Qdrant collection holding **both corpora** under a unified payload schema, embedded with
a retrieval-trained multilingual embedder, exposed to the LLM as a single tool.

**Inputs (produced by the two preprocessing notebooks — download cells at their end):**
- `pdf_chunks_final.jsonl` — 7,136 structure-aware PDF chunks (from `04_pdfs_rag_eda.ipynb`)
- `kcc_chunks_rag.jsonl` (or `.jsonl.gz`) — 716,287 KCC Q&A chunks (from `04_kcc_preprocessing.ipynb`)

**Pipeline:**
1. **Unified ingestion** — both corpora normalized into one schema (`source_type` = `pdf` | `kcc`):
   per-chunk language re-detection (KCC's M2 tag covered only the query half), shared
   district/crop canonicalization, deterministic `chunk_id` for both corpora, token-budget
   enforcement (oversize KCC Hindi chunks re-split at sentence/danda boundaries)
2. **Embedder** — `BAAI/bge-m3` behind an **in-domain ranking gate**: five distinct
   farm topics must rank correctly across EN/Hindi/Hinglish or the notebook refuses to build
3. **Vector DB** — Qdrant (local, on-disk), HNSW `m=16, ef_construct=128`, cosine on
   L2-normalized vectors, payload keyword indexes for filtered retrieval
4. **LLM tool** — `search_agri_knowledge(...)`: JSON-in/JSON-out, never raises, per-source
   weighted fusion (policy intent -> PDF-heavy, field-practice -> KCC-heavy), Milestone-1
   relevance tiers (grounded / fallback / abstain), page-level PDF citations
5. **Evaluation** — farmer-query battery (EN / Devanagari / Hinglish, filters, PDF-vs-KCC
   routing, off-domain abstention) with per-query latency

> **Colab free tier (T4):** embedding is the GPU stage. Full KCC (716k chunks) embeds in
> roughly 20-40 min; set `KCC_MAX_CHUNKS` in the config cell for a faster stratified subset.
> The Qdrant store is written under `RAG_DB_DIR` — copy it to Drive before the runtime dies.

In [1]:
# --- Get the zipped data folder from Drive into Colab ---
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile

ZIP_ON_DRIVE = "/content/drive/MyDrive/data.zip"   # <-- your zip's path in Drive
EXTRACT_TO   = "/content"                          # data/ will appear at /content/data

assert os.path.exists(ZIP_ON_DRIVE), f"not found: {ZIP_ON_DRIVE}  (check the exact name/path in Drive)"
print(f"zip size: {os.path.getsize(ZIP_ON_DRIVE)/1e6:.0f} MB")

with zipfile.ZipFile(ZIP_ON_DRIVE) as zf:
    bad = zf.testzip()            # integrity check — None means the zip is intact
    assert bad is None, f"corrupted zip, first bad file: {bad}"
    zf.extractall(EXTRACT_TO)

# show what landed + verify the KCC chunks are complete
for root, _, files in os.walk("/content/data/final"):
    for fn in files:
        p = os.path.join(root, fn)
        print(f"{os.path.getsize(p)/1e6:8.1f} MB  {p}")

Mounted at /content/drive
zip size: 400 MB
   463.3 MB  /content/data/final/kcc/kcc_chunks_rag.jsonl
     0.0 MB  /content/data/final/kcc/metadata_schema.json
    10.0 MB  /content/data/final/pdfs/pdf_chunks_final.jsonl


In [2]:
# Dependencies. torch ships with Colab; sentencepiece backs the XLM-R (e5) tokenizer.
!pip install qdrant-client sentence-transformers transformers sentencepiece tqdm --break-system-packages -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 12.7 MB/s eta 0:00:00


## 0. Qdrant Server (Colab, no Docker)

`QdrantClient(path=...)` runs Qdrant in **local mode**, which silently ignores the HNSW config and every payload index, and answers each query with a full O(n) scan. Measured on this corpus: **p50 1.6 s at 47k chunks, 3.7 s at 107k** — exactly linear, extrapolating to **~25 s per query at the full 723k**, against a 200–300 ms design target (M3 §12.3).

Colab cannot run Docker (no daemon), but Qdrant ships a **single static Linux binary**, so the cell below downloads it and runs it as a background process on `:6333`. From that point the `HnswConfigDiff(m=16, ef_construct=128)` and the eleven payload indexes in the build cell stop being dead code.

> The server dies with the runtime. Section 8 (EXPORT) snapshots it to Drive; section 9 > (RESTORE) brings it back without re-embedding.

In [3]:
# ---------------------------------------------------------------------------
# Qdrant in SERVER mode, inside Colab. No Docker (Colab has no daemon) — Qdrant
# ships a single static Linux binary, so we just download and run it.
#
# WHY THIS MATTERS: QdrantClient(path=...) is "local mode", which silently
# IGNORES the HNSW config and every payload index, and brute-forces each query.
# Measured: p50 1.6s at 47k chunks, 3.7s at 107k — perfectly linear, i.e. a full
# O(n) scan. Extrapolated to the full 723k corpus that is ~25s per query against
# a 200-300ms design target. Server mode builds the real HNSW graph and makes the
# payload indexes function, which is the only way to serve this corpus.
#
# MUSL, NOT GNU: the linux-gnu asset is dynamically linked and needs GLIBC_2.38,
# while Colab ships glibc 2.35 -- it downloads fine and then dies instantly with
#   /content/qdrant: /lib/x86_64-linux-gnu/libc.so.6: version `GLIBC_2.38' not found
# The linux-musl asset is statically linked and depends on no system libc at all.
# ---------------------------------------------------------------------------
import os, subprocess, tarfile, time, requests

QDRANT_STORAGE = "/content/qdrant_server_storage"
QDRANT_URL     = "http://localhost:6333"
QDRANT_BIN     = "/content/qdrant"
os.makedirs(QDRANT_STORAGE, exist_ok=True)

def qdrant_alive(url=QDRANT_URL, timeout=1):
    try:
        return requests.get(f"{url}/readyz", timeout=timeout).ok
    except Exception:
        return False

def binary_ok(path=QDRANT_BIN):
    """Existence is not enough -- a glibc-linked build downloads fine and only
    fails at exec. Actually run it, so a stale bad binary gets replaced rather
    than silently retried forever."""
    if not os.path.exists(path):
        return False
    try:
        return subprocess.run([path, "--version"], capture_output=True,
                              timeout=60).returncode == 0
    except Exception:
        return False

if qdrant_alive():
    print("Qdrant server already running on :6333")
else:
    if not binary_ok():
        if os.path.exists(QDRANT_BIN):
            print("existing qdrant binary is not runnable here — replacing it")
            os.remove(QDRANT_BIN)
        rel = requests.get("https://api.github.com/repos/qdrant/qdrant/releases/latest",
                           timeout=30).json()
        # musl first (static, no libc dependency); gnu only as a last resort.
        asset = None
        for suffix in ("x86_64-unknown-linux-musl.tar.gz",
                       "x86_64-unknown-linux-gnu.tar.gz"):
            asset = next((a for a in rel["assets"] if a["name"].endswith(suffix)), None)
            if asset:
                break
        if asset is None:
            raise RuntimeError(f"no linux x86_64 asset in {rel['tag_name']}: "
                               + ", ".join(a["name"] for a in rel["assets"]))
        print(f"downloading qdrant {rel['tag_name']} ({asset['name']}) ...")
        with open("/content/q.tar.gz", "wb") as f:
            f.write(requests.get(asset["browser_download_url"], timeout=600).content)
        with tarfile.open("/content/q.tar.gz") as t:
            try:
                t.extractall("/content", filter="data")   # py>=3.12 silences the warning
            except TypeError:
                t.extractall("/content")
        os.chmod(QDRANT_BIN, 0o755)
        if not binary_ok():
            raise RuntimeError("downloaded qdrant still will not execute. Check:\n"
                               + subprocess.run([QDRANT_BIN, "--version"],
                                                capture_output=True).stderr.decode()[-800:])
    print("binary OK:", subprocess.run([QDRANT_BIN, "--version"],
                                       capture_output=True).stdout.decode().strip())

    env = dict(os.environ,
               QDRANT__STORAGE__STORAGE_PATH=QDRANT_STORAGE,
               QDRANT__TELEMETRY_DISABLED="true")
    # Detached so it survives this cell finishing; log kept for diagnosis.
    qdrant_proc = subprocess.Popen([QDRANT_BIN], env=env, cwd="/content",
                                   stdout=open("/content/qdrant.log", "w"),
                                   stderr=subprocess.STDOUT)
    for _ in range(90):
        if qdrant_alive():
            break
        if qdrant_proc.poll() is not None:
            raise RuntimeError("qdrant exited early — see /content/qdrant.log:\n"
                               + open("/content/qdrant.log").read()[-1500:])
        time.sleep(1)
    else:
        raise RuntimeError("qdrant did not become ready in 90s — see /content/qdrant.log")
    print("Qdrant server ready on :6333")

_info = requests.get(f"{QDRANT_URL}/").json()
print(f"  version: {_info.get('version')}   storage: {QDRANT_STORAGE}")
print("  NOTE: the server dies with the Colab runtime. Run the EXPORT cell at the")
print("        end before you close the session, or the build is lost.")


downloading qdrant v1.18.3 (qdrant-x86_64-unknown-linux-musl.tar.gz) ...
binary OK: qdrant 1.18.3
Qdrant server ready on :6333
  version: 1.18.3   storage: /content/qdrant_server_storage
  NOTE: the server dies with the Colab runtime. Run the EXPORT cell at the
        end before you close the session, or the build is lost.


In [4]:
import os

In [5]:
# ---------------------------------------------------------------------------
# Configuration — every tunable in one place (values per Milestone-3 report §9)
# ---------------------------------------------------------------------------
import os

# --- Input chunk artifacts (upload here or mount Drive and point at them) ---
# from google.colab import drive; drive.mount('/content/drive')
def _first_existing(candidates):
    for p in candidates:
        if os.path.exists(p) or os.path.exists(p + ".gz"):
            return p
    return candidates[-1]

# Repo layout first (running inside the cloned repo, chunks produced in place),
# /content fallback (Colab: upload the downloaded artifacts there).
PDF_CHUNKS_PATH = _first_existing(["../data/final/pdfs/pdf_chunks_final.jsonl",
                                   "/content/data/final/pdfs/pdf_chunks_final.jsonl"])
KCC_CHUNKS_PATH = _first_existing(["../data/final/kcc/kcc_chunks_rag.jsonl",
                                   "/content/data/final/kcc/kcc_chunks_rag.jsonl"])   # .gz auto-detected

# --- Embedder (M3 report §9.2; multilingual-e5 — see §5.2 and Appendix G) ---
# REPLACED the MuRIL sentence-transformer after measured retrieval failure. That
# checkpoint could not discriminate WITHIN agriculture: median cosine between
# random UNRELATED KCC chunks was 0.966 (27% of unrelated pairs >0.99), giving
# ~1/6 top-1 accuracy on a 6-document probe and ~0.999 scores for everything,
# off-domain queries included. It separated agri-vs-chess but not wheat-vs-mango,
# which is why the old cross-domain sanity probe passed a model useless for
# ranking. multilingual-e5-base is retrieval-trained (not STS/paraphrase) and
# covers Hindi + English; the swap also closes the M3 Appendix-G validation item
# "MuRIL vs multilingual-e5 on code-mixed retrieval".
# The chunk artifacts are UNCHANGED — this is an embedder swap only.
EMBED_MODEL_NAME = "BAAI/bge-m3"
MODEL_TAG        = "bge_m3"   # names the output dir + scorecard
MODEL_MAX_TOKENS = 512      # bge-m3 natively allows 8192; PINNED to 512 in the load cell
                            # so this run truncates identically to the e5 runs (controlled test)
EMBED_DIM_EXPECTED = 1024   # informational; Qdrant sizes itself from EMBED_DIM

# bge-m3 takes NO prefixes -- it was not trained with them. Both are empty strings,
# so embed_doc/embed_query are pass-throughs and the rest of the notebook is
# byte-identical to the e5 variants. Do NOT add "query: "/"passage: " here.
QUERY_PREFIX = ''
DOC_PREFIX   = ''
embed_doc   = lambda t: DOC_PREFIX + t
embed_query = lambda t: QUERY_PREFIX + t

EMBED_BATCH_SIZE = 64       # lowered for a ~560M model on a T4; halve again on CUDA OOM
CHUNK_TOKEN_BUDGET = 400    # re-split target for oversize KCC chunks (matches PDF chunker)

# --- KCC scale control (Colab free tier) ---
# None = index all 716k chunks (~20-40 min embedding on T4, ~2.5 GB RAM in Qdrant).
# An int (e.g. 100_000) takes a crop-stratified sample for faster iteration runs.
KCC_MAX_CHUNKS = None        # None = FULL 716k KCC corpus (production build)

# --- Output ---
# The directory is tagged by scale so a DEV subset index and the eventual FULL
# production index never share a folder. Without this, switching KCC_MAX_CHUNKS
# to None would reuse the old collection AND its shard_progress.json, and the
# resume logic would skip the first shards -> a silently half-empty index.
_SCALE_TAG      = "full" if KCC_MAX_CHUNKS is None else f"dev_{KCC_MAX_CHUNKS // 1000}k"

# --- Embedding cache (the thing that makes a 723k build survivable) -------
# Embedding is the expensive stage: ~723k chunks through a 568M model on a T4.
# If the runtime dies mid-build you must NOT re-pay that GPU time, so each
# shard's vectors are cached as float16 .npy. Point this at Drive and the cache
# outlives the runtime; a resumed run reloads vectors and only re-does the
# (fast) upsert. float16 halves the cache to ~1.5 GB and costs nothing real:
# the vectors are L2-normalised and cosine is insensitive at this precision.
EMB_CACHE_DIR = "/content/drive/MyDrive/rag_emb_cache_bge_m3_full"   # <- Drive = survives
# EMB_CACHE_DIR = f"/content/emb_cache_{MODEL_TAG}_{_SCALE_TAG}"     # <- local = faster, dies
USE_EMB_CACHE = True
RAG_DB_DIR      = (f"/content/rag_outputs_{MODEL_TAG}_{_SCALE_TAG}" if os.path.isdir("/content")
                   else f"../data/final/rag_index_{MODEL_TAG}_{_SCALE_TAG}")
QDRANT_PATH     = os.path.join(RAG_DB_DIR, "qdrant_db")
COLLECTION_NAME = "agri_knowledge"                      # one collection, both corpora
os.makedirs(RAG_DB_DIR, exist_ok=True)

# --- Retrieval (M3 report §9.4-9.6; tiers measured for bge-m3) ---
TOP_K_DEFAULT  = 5

# TIER THRESHOLDS ARE MODEL-SPECIFIC AND ARE NOT TRANSFERABLE.
# bge-m3 uses a completely different cosine scale from the e5 family. Measured on
# the 47k-chunk run: unrelated documents sit at 0.404 median (e5 put them at
# 0.844), in-domain queries land ~0.59-0.79 and off-domain ~0.49-0.52. The
# e5 pair (0.837 / 0.857) is meaningless here -- applied to bge-m3 it would
# abstain on literally every query, including perfect matches.
#
# The pair below is what check D2 derived from bge-m3's own score distribution
# (in-domain p50=0.676 min=0.590 | off-domain p90=0.517 max=0.517), giving the
# first separation margin wide enough for abstention to actually work: +0.072,
# versus +0.018 for e5-large and +0.009 for e5-base.
#
# NOTE these were measured at 47k chunks. This notebook is now set to 100k, and a
# larger index shifts the distribution (more candidates -> better best-match), so
# D2 will re-derive and override them. Treat the values below as the seed, and
# paste D2's new output back here after the 100k run.
TIER_GROUNDED  = 0.638      # >= : cite-and-answer
TIER_FALLBACK  = 0.553      # >= : answer + mandatory "verify with local KVK" disclaimer
                            # <  : abstain / out-of-scope

# Per-source fusion weights by query intent (M2 §7.2 design, validated in eval below):
FUSION_WEIGHTS = {
    "policy":         {"pdf": 2.0, "kcc": 0.5},   # scheme/eligibility/subsidy questions
    "field_practice": {"pdf": 0.5, "kcc": 2.0},   # how-to / dosage / cultivation questions
    "general":        {"pdf": 1.0, "kcc": 1.0},
}

print(f"PDF chunks : {PDF_CHUNKS_PATH}  exists={os.path.exists(PDF_CHUNKS_PATH)}")
print(f"KCC chunks : {KCC_CHUNKS_PATH}  exists={os.path.exists(KCC_CHUNKS_PATH) or os.path.exists(KCC_CHUNKS_PATH + '.gz')}")
print(f"Embedder   : {EMBED_MODEL_NAME}  (prefixes: query={QUERY_PREFIX!r} doc={DOC_PREFIX!r})")
print(f"Scale mode : " + ("PRODUCTION full corpus" if KCC_MAX_CHUNKS is None
                          else f"DEV subset (KCC<={KCC_MAX_CHUNKS:,})"))
print(f"Vector DB  : {QDRANT_PATH}")
print(f"Tiers      : abstain < {TIER_FALLBACK} <= fallback < {TIER_GROUNDED} <= grounded"
      "   (recalibrated by eval check D2)")

PDF chunks : /content/data/final/pdfs/pdf_chunks_final.jsonl  exists=True
KCC chunks : /content/data/final/kcc/kcc_chunks_rag.jsonl  exists=True
Embedder   : BAAI/bge-m3  (prefixes: query='' doc='')
Scale mode : PRODUCTION full corpus
Vector DB  : /content/rag_outputs_bge_m3_full/qdrant_db
Tiers      : abstain < 0.553 <= fallback < 0.638 <= grounded   (recalibrated by eval check D2)


## 1. Unified Ingestion — PDF + KCC Coexistence

Both corpora are normalized into **one payload schema** (M3 report Appendix-B design). The three
measured schema conflicts between the corpora are resolved here, at ingestion:

| Conflict | Resolution |
|---|---|
| KCC `language` described only the *query* half (`english` for 98.8%-Devanagari answers) | `language` re-detected **per chunk** over the full text, same detector for both corpora |
| KCC districts are raw uppercase with typos (`KANPUR CITY`, `MAHARAHGANJ`); PDF uses canonical post-bifurcation names | one shared canonicalization map at ingestion; raw value preserved in `district_raw` |
| KCC chunks have no deterministic identity | `chunk_id = uuid5(content-hash : chunk_number)` for KCC; PDF chunks already carry `uuid5(sha256 : index)` |

Additionally: KCC chunks over the 512-token embedder budget (the M2 chunker was character-based
and its sentence splitter missed the Devanagari danda) are **re-split here** at sentence/danda
boundaries to the same 400-token budget the PDF chunker uses. The 98.5% single-chunk majority
passes through with field mapping only.

**Unified fields (every chunk):** `chunk_id, source_type, source, text, language, year, crop,
district, chunk_index, n_chunks_in_doc`.
**PDF-only:** `filename, doc_category, heading_hierarchy, page_start, page_end, has_table,
extraction_method, source_pdf_sha256`. **KCC-only:** `season, query_type, category, month,
district_raw`. A filter on a per-corpus field implicitly restricts to that corpus.

In [6]:
import gzip
import hashlib
import json
import re
import uuid
from collections import Counter, defaultdict

from tqdm.auto import tqdm

# ---------------------------------------------------------------------------
# Shared normalization helpers (used by BOTH corpora — this is the coexistence
# contract: same language detector, same district/crop canonical vocabulary)
# ---------------------------------------------------------------------------

def detect_language(text, sample_chars=1000):
    """Per-chunk script-ratio language tag: en / hi / mixed.
    Same thresholds as the PDF EDA notebook, so tags stay comparable."""
    s = text[:sample_chars]
    dev = len(re.findall(r"[\u0900-\u097F]", s))
    lat = len(re.findall(r"[a-zA-Z]", s))
    total = max(dev + lat, 1)
    if dev / total > 0.15 and lat / total > 0.15:
        return "mixed"
    return "hi" if dev / total > 0.3 else "en"

# District renames/typos -> canonical post-bifurcation names (extends the
# VALIDATED_ALIASES map from 04_pdfs_rag_eda.ipynb; KCC adds raw-source typos).
DISTRICT_CANON = {
    "allahabad": "prayagraj", "faizabad": "ayodhya",
    "prabuddh nagar": "shamli", "prabudh nagar": "shamli",
    "bhim nagar": "sambhal", "panchsheel nagar": "hapur",
    "jyotiba phule nagar": "amroha", "jyotibaphule nagar": "amroha",
    "kanshi ram nagar": "kasganj", "kanshiram nagar": "kasganj",
    "chhatrapati shahuji maharaj nagar": "amethi",
    "mahamaya nagar": "hathras", "ramabai nagar": "kanpur dehat",
    "banaras": "varanasi", "kashi": "varanasi",
    # raw KCC source typos observed in the corpus
    "kanpur city": "kanpur nagar", "maharahganj": "maharajganj",
    "sant ravidas nagar": "bhadohi",
}

def canon_district(raw):
    """Lowercase, collapse spaces, apply the shared rename/typo map."""
    if not raw or str(raw).lower() in ("unknown", "nan", "none", ""):
        return None
    d = re.sub(r"\s+", " ", str(raw).strip().lower())
    return DISTRICT_CANON.get(d, d)

# Canonical crop vocabulary. Keys are ANY surface form (English name, Hindi /
# Hinglish vernacular, or an alias appearing inside the KCC parentheses); values
# are the single canonical token stored in the payload AND accepted by the tool's
# `crop` filter. Every canonical value is also a key, so canon_crop() is
# idempotent -- payload value and user filter term must converge on one string.
CROP_CANON = {
    "rice": "rice", "paddy": "rice", "dhan": "rice", "chawal": "rice",
    "wheat": "wheat", "gehun": "wheat", "gehu": "wheat", "kanak": "wheat",
    "maize": "maize", "makka": "maize", "makai": "maize", "bhutta": "maize", "corn": "maize",
    "sugarcane": "sugarcane", "ganna": "sugarcane", "noble cane": "sugarcane",
    "mustard": "mustard", "sarson": "mustard", "raya": "mustard",
    "indian mustard": "mustard", "indian rapeseed and mustard": "mustard", "yellow sarson": "mustard",
    "urad": "urad", "black gram": "urad", "urd": "urad", "urd bean": "urad",
    "gram": "gram", "bengal gram": "gram", "chana": "gram", "chick pea": "gram", "kabuli": "gram",
    "moong": "moong", "green gram": "moong", "moong bean": "moong", "mung": "moong",
    "arhar": "arhar", "pigeon pea": "arhar", "red gram": "arhar", "tur": "arhar",
    "masur": "masur", "lentil": "masur",
    "okra": "okra", "bhindi": "okra", "ladysfinger": "okra",
    "bajra": "bajra", "pearl millet": "bajra", "bulrush millet": "bajra", "spiked millet": "bajra",
    "jowar": "jowar", "sorghum": "jowar", "great millet": "jowar",
    "barley": "barley", "jau": "barley",
    "sesame": "sesame", "til": "sesame", "gingelly": "sesame", "sesamum": "sesame",
    "groundnut": "groundnut", "pea nut": "groundnut", "peanut": "groundnut", "mung phalli": "groundnut",
    "colocasia": "arvi", "arvi": "arvi", "arbi": "arvi", "arum": "arvi",
    "cotton": "cotton", "kapas": "cotton",
    "soybean": "soybean", "bhat": "soybean",
    "linseed": "linseed", "alsi": "linseed",
    "spinach": "spinach", "palak": "spinach",
    "methi": "fenugreek", "fenugreek": "fenugreek",
    "rajma": "rajma", "french bean": "rajma",
    "sunflower": "sunflower", "suryamukhi": "sunflower",
    "finger millet": "ragi", "fingermillet": "ragi", "ragi": "ragi", "mandika": "ragi",
    "pea": "pea", "peas": "pea", "matar": "pea", "field peas": "pea", "garden peas": "pea",
}

_CROP_PAREN = re.compile(r"^([^(]+?)\s*\((.*)\)\s*$")
_CROP_NULLS = ("unknown", "nan", "none", "", "na", "n/a", "other", "others")

def canon_crop(raw):
    """Normalise ANY crop surface form to one canonical token.

    KCC stores crops as 'Canonical (vernacular/alias/alias)' -- 'Paddy (Dhan)',
    'Maize (Makka)', 'Bengal Gram (Gram/Chick Pea/Chana)'. The previous version
    only matched bare names, so the parenthesised form never hit CROP_CANON:
    crop='rice' resolved to 'rice' while 112,266 chunks sat under the payload
    value 'paddy (dhan)'. Filtering rice or maize matched ZERO chunks -- silently,
    with no error (this is what eval check A2 was actually reporting).

    Order: whole string -> base before '(' -> each alias inside '()' -> the base.
    Applied identically to payload values at ingestion and to the caller's filter
    term at query time, so both sides always land on the same token.
    """
    if raw is None:
        return None
    c = re.sub(r"\s+", " ", str(raw).strip().lower())
    if c in _CROP_NULLS:
        return None
    if c in CROP_CANON:                          # 'rice', 'dhan', 'paddy'
        return CROP_CANON[c]
    m = _CROP_PAREN.match(c)
    if m:
        base, inner = m.group(1).strip(), m.group(2)
        if base in CROP_CANON:                   # 'paddy (dhan)' -> 'paddy' -> 'rice'
            return CROP_CANON[base]
        for alias in re.split(r"[/,]", inner):   # 'bhindi(okra/ladysfinger)'
            alias = alias.strip()
            if alias in CROP_CANON:
                return CROP_CANON[alias]
        return base                              # unmapped: keep the English base
    return c

def read_jsonl(path):
    """Stream a .jsonl or .jsonl.gz line-by-line."""
    if not path.endswith(".gz") and not os.path.exists(path) and os.path.exists(path + ".gz"):
        path = path + ".gz"
    opener = gzip.open if path.endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

# ---------------------------------------------------------------------------
# Load PDF chunks -> unified schema (metadata already rich; mostly a re-map)
# ---------------------------------------------------------------------------
unified = []

for c in tqdm(read_jsonl(PDF_CHUNKS_PATH), desc="PDF chunks"):
    m = c["metadata"]
    filename = m.get("filename", "")
    # ACP files are district contingency plans named after the district
    district = canon_district(os.path.splitext(filename)[0]) if m.get("source") == "up_acp" else None
    unified.append({
        # shared
        "chunk_id": m["chunk_id"],
        "source_type": "pdf",
        "source": m.get("source", ""),
        "text": c["text"],
        "language": m.get("detected_language") or detect_language(c["text"]),
        "year": int(m["detected_year"]) if m.get("detected_year") else None,
        "crop": None,                      # document-level crop tagging: future work
        "district": district,
        "chunk_index": m.get("chunk_index", 0),
        "n_chunks_in_doc": m.get("n_chunks_in_doc", 1),
        # pdf-only
        "filename": filename,
        "doc_category": m.get("doc_category", ""),
        "heading_hierarchy": m.get("heading_hierarchy", ""),
        "page_start": m.get("page_start"),
        "page_end": m.get("page_end"),
        "has_table": bool(m.get("has_table", False)),
        "extraction_method": m.get("extraction_method", ""),
        "source_pdf_sha256": m.get("source_pdf_sha256", ""),
    })

n_pdf = len(unified)
print(f"PDF chunks loaded: {n_pdf:,}")
print(f"  with page provenance: {sum(1 for u in unified if u['page_start'] is not None):,}")
print(f"  with district (ACP):  {sum(1 for u in unified if u['district']):,}")
print(f"  language mix: {Counter(u['language'] for u in unified)}")

PDF chunks: 0it [00:00, ?it/s]

PDF chunks loaded: 7,136
  with page provenance: 7,117
  with district (ACP):  4,818
  language mix: Counter({'en': 7052, 'mixed': 84})


In [7]:
# ---------------------------------------------------------------------------
# Load + normalize KCC chunks -> unified schema
#   - deterministic chunk_id (uuid5 over content hash)
#   - per-chunk language re-detection (full Q+A text, not just the query)
#   - shared district/crop canonicalization
#   - token-budget enforcement: oversize chunks re-split at sentence/danda
# ---------------------------------------------------------------------------
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)

def n_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

SENT_SPLIT = re.compile(r"(?<=[.!?।])\s+")   # । = Devanagari danda

def split_to_budget(text, budget=CHUNK_TOKEN_BUDGET):
    """Sentence-accumulating re-split for the oversize tail (danda-aware).
    Only called for chunks over the embedder budget — ~1.5% of the corpus."""
    sents = SENT_SPLIT.split(text)
    parts, cur, cur_tok = [], [], 0
    for s in sents:
        t = n_tokens(s)
        if cur and cur_tok + t > budget:
            parts.append(" ".join(cur))
            cur, cur_tok = [], 0
        cur.append(s)
        cur_tok += t
    if cur:
        parts.append(" ".join(cur))
    return parts or [text]

def kcc_chunk_id(text, meta, chunk_no, seq):
    # `seq` = source-record index in the file. KCC dedup removed exact
    # (query, answer, crop) duplicates, but near-duplicate call records sharing
    # the same leading text + crop/district/time survive; hashing full text plus
    # the record index guarantees a unique id for every chunk while staying
    # deterministic for a given input file (so re-ingestion is idempotent).
    basis = "|".join([text, str(meta.get("crop")), str(meta.get("district")),
                      str(meta.get("year")), str(meta.get("month")),
                      str(seq), str(chunk_no)])
    h = hashlib.sha1(basis.encode("utf-8")).hexdigest()
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"kcc:{h}"))

kcc_rows, n_resplit, n_seen = [], 0, 0
budget_hard = MODEL_MAX_TOKENS - 2

for c in tqdm(read_jsonl(KCC_CHUNKS_PATH), desc="KCC chunks"):
    n_seen += 1
    m = c.get("metadata", {})
    text = c.get("text", "")
    if len(text.strip()) < 10:
        continue

    # cheap length pre-filter: only tokenize candidates that could exceed budget
    pieces = [text]
    if len(text) > 900:                          # ~ >250 tokens; safe lower bound
        if n_tokens(text) > budget_hard:
            pieces = split_to_budget(text)
            n_resplit += 1

    year = m.get("year")
    base = {
        "source_type": "kcc",
        "source": "kcc_qa",
        "language": detect_language(text),
        "year": int(year) if year else None,
        "crop": canon_crop(m.get("crop")),
        "district": canon_district(m.get("district")),
        "district_raw": m.get("district"),
        "season": m.get("season") if m.get("season") not in (None, "unknown") else None,
        "query_type": m.get("query_type"),
        "category": m.get("category"),
        "month": int(m["month"]) if m.get("month") else None,
    }
    for i, piece in enumerate(pieces):
        row = dict(base)
        row["text"] = piece
        row["chunk_index"] = i if len(pieces) > 1 else int(c.get("chunk_number", 1)) - 1
        row["n_chunks_in_doc"] = len(pieces) if len(pieces) > 1 else int(c.get("total_chunks", 1))
        row["chunk_id"] = kcc_chunk_id(piece, m, row["chunk_index"], n_seen)
        kcc_rows.append(row)

print(f"KCC chunks read: {n_seen:,} -> normalized rows: {len(kcc_rows):,} "
      f"(oversize re-split: {n_resplit:,})")

# --- Optional crop-stratified subsample for fast Colab iteration -------------
if KCC_MAX_CHUNKS and len(kcc_rows) > KCC_MAX_CHUNKS:
    import random
    random.seed(42)
    by_crop = defaultdict(list)
    for r in kcc_rows:
        by_crop[r["crop"] or "_none"].append(r)
    frac = KCC_MAX_CHUNKS / len(kcc_rows)
    sampled = []
    for crop, rows in by_crop.items():
        k = max(1, round(len(rows) * frac))
        sampled.extend(random.sample(rows, min(k, len(rows))))
    kcc_rows = sampled[:KCC_MAX_CHUNKS]
    print(f"Stratified subsample applied: {len(kcc_rows):,} chunks "
          f"across {len(by_crop)} crops (KCC_MAX_CHUNKS={KCC_MAX_CHUNKS:,})")

unified.extend(kcc_rows)
print(f"\nUnified corpus: {len(unified):,} chunks "
      f"(pdf={n_pdf:,}, kcc={len(unified) - n_pdf:,})")

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

KCC chunks: 0it [00:00, ?it/s]

KCC chunks read: 716,287 -> normalized rows: 716,303 (oversize re-split: 16)

Unified corpus: 723,439 chunks (pdf=7,136, kcc=716,303)


In [8]:
# ---------------------------------------------------------------------------
# Corpus summary — the coexistence picture at a glance
# ---------------------------------------------------------------------------
# Counter-based, NOT pandas. `pd.DataFrame(unified)` materialises a second full
# copy of the corpus; at 723k chunks that is a gratuitous multi-GB spike right
# before the embedding stage, and it was a contributor to the original OOM.
from collections import Counter

by_source = Counter(u["source_type"] for u in unified)
by_lang   = Counter((u["source_type"], u["language"]) for u in unified)
by_dist   = Counter(u["district"] for u in unified if u.get("district"))
by_crop   = Counter(u["crop"] for u in unified if u["source_type"] == "kcc" and u.get("crop"))

print(f"Total chunks: {len(unified):,}\n")
print("By source_type:")
for k, v in by_source.most_common():
    print(f"  {k:<5} {v:>9,}")

print("\nLanguage (re-detected per chunk — note KCC is mostly `mixed`/`hi`,")
print("not the `english` the M2 query-only tag suggested):")
for (st, lg), v in sorted(by_lang.items()):
    print(f"  {st:<5} {lg:<6} {v:>9,}")

print("\nTop districts after shared canonicalization:")
for k, v in by_dist.most_common(8):
    print(f"  {k:<22} {v:>9,}")

print("\nTop crops after canonicalization (KCC):")
for k, v in by_crop.most_common(8):
    print(f"  {k:<22} {v:>9,}")

# Identity check without building a DataFrame: a set of ids is ~50 MB at 723k.
_ids = set()
_dup = 0
for u in unified:
    if u["chunk_id"] in _ids:
        _dup += 1
    else:
        _ids.add(u["chunk_id"])
assert _dup == 0, f"{_dup} duplicate chunk_ids — deterministic identity broken"
print(f"\nchunk_id uniqueness: OK ({len(_ids):,} unique)")
del _ids


Total chunks: 723,439

By source_type:
  kcc     716,303
  pdf       7,136

Language (re-detected per chunk — note KCC is mostly `mixed`/`hi`,
not the `english` the M2 query-only tag suggested):
  kcc   en         9,522
  kcc   hi         8,498
  kcc   mixed    698,283
  pdf   en         7,052
  pdf   mixed         84

Top districts after shared canonicalization:
  sitapur                   25,248
  bareilly                  20,669
  kheri                     19,729
  bulandshahar              19,348
  badaun                    19,319
  aligarh                   19,246
  hardoi                    17,969
  jaunpur                   17,245

Top crops after canonicalization (KCC):
  rice                     112,269
  wheat                     95,836
  sugarcane                 58,646
  potato                    40,818
  mustard                   39,147
  mango                     36,809
  pea                       19,993
  chillies                  18,187

chunk_id uniqueness: OK (723,439

## 2. Embedding Model — BAAI/bge-m3 (gated on in-domain ranking)

XLM-RoBERTa large backbone (568M), 1024-dim, trained specifically for **multilingual
retrieval** across 100+ languages with hard negatives. Uses **CLS pooling**, and takes
**no prefixes at all** -- adding e5's `query: `/`passage: ` here would inject meaningless
tokens into every vector, so both prefixes are empty strings below.

Its native limit is 8192 tokens, but `max_seq_length` is pinned to 512 in the load cell
so this run truncates text identically to the two e5 runs. This is a controlled
comparison -- letting bge-m3 read longer chunks would change the inputs, not just the
model, and would also blow up T4 memory.

**Why this comparison exists.** The original embedder (`Yunika/muril-base-sentence-transformer`) was measured unusable on this corpus: unrelated
KCC chunks scored **0.966** median cosine, top-1 accuracy was **1/6** on a six-document probe,
and a wheat-urea query returned capsicum advice at `0.999` confidence. `multilingual-e5-base`
fixed retrieval but left thin margins (as low as **+0.006**), a **+0.009** in/out-of-domain
separation, and failing Hinglish. This notebook tests whether a stronger model closes that gap.

**The load gate is the decisive check.** The old probe compared "rice blast" against "how to
win at chess" — a *cross-domain* contrast that a collapsed model passes easily. The gate below
runs a real **ranking** task over five distinct farm topics and hard-fails on any of three
conditions: fewer than 4/5 gold documents ranked first, a mean gold margin under 0.01, or
unrelated documents above 0.97 cosine.

In [9]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else " — embedding will be slow; enable the GPU runtime"))

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
# Pin the sequence length so ALL model notebooks truncate text identically.
# This is a controlled comparison: only the model may vary, never the inputs.
# (bge-m3 defaults to 8192, which would both change what each chunk contributes
#  and exhaust T4 memory.)
embed_model.max_seq_length = MODEL_MAX_TOKENS
EMBED_DIM = embed_model.get_sentence_embedding_dimension()
print(f"Model: {EMBED_MODEL_NAME} | dim={EMBED_DIM}")

# --- Config report (informational, NOT a hard gate) -------------------------
# The previous version asserted max_seq_length==512 AND mean pooling, which
# hard-crashes on perfectly good alternatives (bge-m3 is CLS-pooled with an
# 8192 limit). Config shape was never the real risk — ranking quality was.
print("\nModule stack:")
for name, module in embed_model.named_children():
    print(f"  ({name}): {module}")
if embed_model.max_seq_length != MODEL_MAX_TOKENS:
    print(f"[WARN] max_seq_length={embed_model.max_seq_length} but MODEL_MAX_TOKENS={MODEL_MAX_TOKENS}; "
          "chunk token-budget enforcement used the latter — align them if they diverge a lot.")
if EMBED_DIM != EMBED_DIM_EXPECTED:
    print(f"[INFO] dim={EMBED_DIM} (expected {EMBED_DIM_EXPECTED} for e5-base) — "
          "fine, Qdrant sizes the collection from EMBED_DIM.")

# ---------------------------------------------------------------------------
# IN-DOMAIN RETRIEVAL GATE  (this is the check that actually matters)
# ---------------------------------------------------------------------------
# History: the earlier gate compared "rice blast" against "how to win at chess"
# and passed with margin +0.874 — while the model was scoring UNRELATED farm
# chunks at 0.966 median and returning capsicum advice for a wheat-urea query.
# A cross-domain contrast cannot detect that. Retrieval needs agri-vs-agri
# resolution, so the gate below is a real ranking task over distinct farm
# topics, mixing EN / Devanagari / Hinglish queries against Hindi + English docs.
GATE_DOCS = [
    ("wheat-urea",    "गेहूं में यूरिया की टॉप ड्रेसिंग 50 किलो प्रति एकड़ पहली सिंचाई के बाद करें"),
    ("wheat-rust",    "गेहूं में पीला रतुआ रोग के लिए प्रोपिकोनाजोल 25 ईसी 200 मिली प्रति एकड़ छिड़काव करें"),
    ("paddy-nursery", "धान की नर्सरी की बुवाई जून के पहले पखवाड़े में करें"),
    ("pmkisan",       "PM-KISAN provides income support of Rs 6000 per year to landholding farmer families in three instalments"),
    ("mango-hopper",  "आम के पेड़ में भुनगा कीट के लिए इमिडाक्लोप्रिड 0.5 मिली प्रति लीटर पानी छिड़कें"),
]
GATE_QUERIES = [   # (gold doc, query, script) — deliberately cross-script
    ("wheat-urea",    "how much urea to apply in wheat top dressing", "EN->HI"),
    ("wheat-rust",    "gehu me pila ratua ki dawa batao",             "HING->HI"),
    ("paddy-nursery", "धान की नर्सरी कब बोनी चाहिए",                    "HI->HI"),
    ("pmkisan",       "pm kisan samman nidhi eligibility and benefit", "EN->EN"),
    ("mango-hopper",  "आम में भुनगा कीट का उपचार",                      "HI->HI"),
]

_labels = [l for l, _ in GATE_DOCS]
_D = embed_model.encode([embed_doc(t) for _, t in GATE_DOCS], normalize_embeddings=True)
_Q = embed_model.encode([embed_query(q) for _, q, _ in GATE_QUERIES], normalize_embeddings=True)
_S = _Q @ _D.T

print("\nIN-DOMAIN RANKING GATE (gold doc must rank #1):")
_correct, _margins = 0, []
for i, (gold, q, script) in enumerate(GATE_QUERIES):
    order = np.argsort(-_S[i])
    top, gi = _labels[order[0]], _labels.index(gold)
    ok = (top == gold); _correct += ok
    margin = float(_S[i][gi] - max(_S[i][j] for j in range(len(_labels)) if j != gi))
    _margins.append(margin)
    print(f"  [{'OK ' if ok else 'BAD'}] {script:<9} {q[:38]:<40} -> {top:<14} "
          f"gold_score={_S[i][gi]:.3f} margin={margin:+.3f}")

# Anisotropy indicator: how similar are UNRELATED docs to each other?
# Reported, not gated — e5 compresses its range, so a high-ish absolute value
# is normal here; the ranking result above is the decisive signal.
_off = (_D @ _D.T)[~np.eye(len(GATE_DOCS), dtype=bool)]
print(f"\n  unrelated doc-doc cosine: median={np.median(_off):.3f} max={_off.max():.3f}")
print(f"  mean gold margin: {np.mean(_margins):+.3f}   ranking: {_correct}/{len(GATE_QUERIES)}")

# Three INDEPENDENT failure conditions. Ranking alone is not enough: on a 5-item
# set a collapsed model can still land the diagonal by luck, so a near-zero gold
# margin, or near-1.0 similarity between UNRELATED docs, fails the build outright.
_fail = []
if _correct < 4:
    _fail.append(f"ranking {_correct}/{len(GATE_QUERIES)} correct (need >=4)")
if np.mean(_margins) < 0.01:
    _fail.append(f"mean gold margin {np.mean(_margins):+.4f} (need >=0.01) — the gold doc "
                 "barely beats unrelated ones, so the ordering is noise")
if np.median(_off) > 0.97:
    _fail.append(f"unrelated doc-doc cosine median {np.median(_off):.3f} (need <=0.97) — "
                 "the embedding space is collapsed")
if _fail:
    raise RuntimeError(
        "IN-DOMAIN RETRIEVAL GATE FAILED:\n  - " + "\n  - ".join(_fail) +
        "\nThis model cannot rank distinct agricultural topics apart — the exact failure that "
        "made the MuRIL sentence-transformer unusable (unrelated KCC chunks at 0.966 median; "
        "capsicum advice returned for a wheat-urea query). DO NOT BUILD THE INDEX: every "
        "downstream score, tier and eval number would be meaningless. Swap EMBED_MODEL_NAME "
        "(try BAAI/bge-m3 or Alibaba-NLP/gte-multilingual-base) and re-run this cell.")
if np.mean(_margins) < 0.03:
    print("[WARN] gold margins are thin — ranking is correct but fragile; expect unstable "
          "ordering on real queries and check the eval + inspector cells below.")
if np.median(_off) > 0.92:
    print(f"[WARN] unrelated docs sit at {np.median(_off):.3f} cosine — high. Watch the "
          "off-domain separation reported by eval check D1.")
print("\nGate PASSED — safe to build the index.")

Device: cuda (Tesla T4)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

/tmp/ipykernel_406/717150476.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBED_DIM = embed_model.get_sentence_embedding_dimension()


Model: BAAI/bge-m3 | dim=1024

Module stack:
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({})

IN-DOMAIN RANKING GATE (gold doc must rank #1):
  [OK ] EN->HI    how much urea to apply in wheat top dr   -> wheat-urea     gold_score=0.675 margin=+0.182
  [OK ] HING->HI  gehu me pila ratua ki dawa batao         -> wheat-rust     gold_score=0.502 margin=+0.122
  [OK ] HI->HI    धान की नर्सरी कब बोनी चाहिए              -> paddy-nursery  gold_score=0.778 margin=+0.318
  [OK ] EN->EN    pm kisan samman nidhi eligibility and    -> pmkisan        gold_score=0.526 margin=+0.203
  [OK ] HI->HI    आम में भुनगा कीट का उपचार                -> mango-hopper   gold_score=0.728 margin=+0.285

  unrelated doc-

In [10]:
# ---------------------------------------------------------------------------
# Embedding is STREAMED shard-by-shard during indexing (next cell), so the full
# embedding array is NEVER held in RAM at once. Holding the whole array WHILE
# Qdrant builds the HNSW index is what OOM-crashed the one-shot version. Here we
# only set the shard size; the model is already loaded and sanity-checked above.
# The corpus size is whatever the config cell selected (DEV subset or full) —
# every chunk in `unified` is indexed; nothing is dropped at this stage.
# ---------------------------------------------------------------------------
SHARD_SIZE = 20_000   # checkpoint granularity. Smaller = less work lost when a
                      # Colab runtime dies mid-build. At 723k this is ~37 shards;
                      # 50k shards meant losing up to 50k chunks of GPU time per crash.
_n = (len(unified) + SHARD_SIZE - 1) // SHARD_SIZE
print(f"Corpus: {len(unified):,} chunks -> {_n} shards of up to {SHARD_SIZE:,}")
print("Embedding runs shard-by-shard in the build cell below (peak RAM = one shard).")


Corpus: 723,439 chunks -> 37 shards of up to 20,000
Embedding runs shard-by-shard in the build cell below (peak RAM = one shard).


## 3. Qdrant — Sharded Build, HNSW + Cosine, Payload Indexes

One `agri_knowledge` collection for both corpora (M3 report Sec 5.5 / 9.3-9.4): HNSW
`m=16, ef_construct=128`, cosine on L2-normalized vectors (dim from the embedder), on-disk vectors,
payload keyword indexes for filtered retrieval.

**Full corpus, memory-safe build.** Embedding + upsert run in **shards of ~50k** — each
shard is embedded, upserted, then freed, so peak RAM is one shard (~150 MB) instead of
the whole 2.2 GB array. HNSW indexing is **deferred** (`indexing_threshold=0` during
upload, flipped on at the end) so it builds once rather than thrashing during insert.
A `shard_progress.json` checkpoint makes the build **resumable** after an interruption.
No chunks are dropped — every chunk in `unified` is indexed, at whatever scale
`KCC_MAX_CHUNKS` selected. The store lives under a scale-tagged `RAG_DB_DIR`, so a
DEV index and the FULL production index can never overwrite each other.

> Point IDs are the deterministic `chunk_id`s, so re-running is idempotent and resumable.
> `indexing_threshold` must be POSITIVE to build HNSW; `0` (used only during upload here)
> disables it.


In [11]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, HnswConfigDiff, OptimizersConfigDiff, PointStruct, VectorParams,
)
import gc
import json
import time

import numpy as np
from tqdm.auto import tqdm

qdrant_client = QdrantClient(url=QDRANT_URL, timeout=600)   # SERVER mode
# Server mode is what makes the HnswConfigDiff and the payload indexes below
# real; in local mode both are silently ignored and every query is an O(n)
# scan. timeout is generous because bulk upserts of 20k points can be slow.
PROGRESS_PATH = os.path.join(RAG_DB_DIR, "shard_progress.json")
if USE_EMB_CACHE:
    # If EMB_CACHE_DIR points into Drive but Drive is NOT mounted, makedirs would
    # happily create a plain local folder at that path -- the cache would look fine
    # and then vanish with the runtime, defeating its entire purpose. Fail loudly.
    if EMB_CACHE_DIR.startswith('/content/drive/') and not os.path.ismount('/content/drive'):
        raise RuntimeError(
            'EMB_CACHE_DIR is on Drive but Drive is not mounted. Run the Drive-mount \n'
            'cell first, or point EMB_CACHE_DIR at /content/... and accept that the \n'
            'cache dies with the runtime.')
    os.makedirs(EMB_CACHE_DIR, exist_ok=True)
    _cached = len([f for f in os.listdir(EMB_CACHE_DIR) if f.endswith('.npy')])
    print(f"embedding cache: {EMB_CACHE_DIR}  ({_cached} shards already embedded)")
n_shards = (len(unified) + SHARD_SIZE - 1) // SHARD_SIZE

# --- Resume an interrupted build, or start clean ---------------------------
done, fresh = set(), True
if qdrant_client.collection_exists(COLLECTION_NAME) and os.path.exists(PROGRESS_PATH):
    done = set(json.load(open(PROGRESS_PATH)))
    fresh = False
    print(f"Resuming build: {len(done)}/{n_shards} shards already uploaded (skipping those)")

if fresh:
    if qdrant_client.collection_exists(COLLECTION_NAME):
        qdrant_client.delete_collection(COLLECTION_NAME)
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE, on_disk=True),
        hnsw_config=HnswConfigDiff(m=16, ef_construct=128),
        # Indexing OFF during bulk upload; built once at the end. This avoids the
        # incremental-index RAM/CPU spike that OOM-crashed the single-shot build.
        optimizers_config=OptimizersConfigDiff(indexing_threshold=0),
    )
    for field in ["source_type", "source", "language", "doc_category",
                  "crop", "district", "season", "query_type"]:
        qdrant_client.create_payload_index(COLLECTION_NAME, field_name=field, field_schema="keyword")
    qdrant_client.create_payload_index(COLLECTION_NAME, field_name="year", field_schema="integer")
    qdrant_client.create_payload_index(COLLECTION_NAME, field_name="has_table", field_schema="bool")
    json.dump([], open(PROGRESS_PATH, "w"))
    print(f"Fresh collection '{COLLECTION_NAME}' created (HNSW indexing deferred)")

# --- Sharded: embed -> upsert -> free. Peak RAM = one shard (~150 MB) ------
UPSERT_BATCH = 256
done_chunks = sum(min((k + 1) * SHARD_SIZE, len(unified)) - k * SHARD_SIZE for k in done)

# Outer bar = overall progress across the whole corpus (with ETA); the embedder's
# own per-batch bar (show_progress_bar=True) shows live progress WITHIN each shard,
# so the long embedding stretches no longer look frozen.
pbar = tqdm(total=len(unified), initial=done_chunks, desc="Indexing corpus", unit="chunk")
for s in range(n_shards):
    if s in done:
        continue
    lo, hi = s * SHARD_SIZE, min((s + 1) * SHARD_SIZE, len(unified))
    shard = unified[lo:hi]
    # --- embed, or reload from cache if a previous run already paid for it ---
    _cache_f = os.path.join(EMB_CACHE_DIR, f"emb_{s:05d}.npy") if USE_EMB_CACHE else None
    if _cache_f and os.path.exists(_cache_f):
        vecs = np.load(_cache_f).astype(np.float32)
        if len(vecs) != len(shard):          # stale cache from a different scale
            raise RuntimeError(f"cache {_cache_f} has {len(vecs)} rows, shard needs "
                               f"{len(shard)} — clear EMB_CACHE_DIR and rebuild")
        pbar.set_postfix_str(f"shard {s + 1}/{n_shards} (cached)")
    else:
        pbar.set_postfix_str(f"shard {s + 1}/{n_shards} (embedding...)")
        vecs = embed_model.encode(
            [embed_doc(r["text"]) for r in shard], batch_size=EMBED_BATCH_SIZE,
            show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True,
        ).astype(np.float32)
        if _cache_f:                          # float16 halves the cache, cosine-safe
            np.save(_cache_f, vecs.astype(np.float16))
    pbar.set_postfix_str(f"shard {s + 1}/{n_shards} (upserting...)")
    for b in range(0, len(shard), UPSERT_BATCH):
        pts = [
            PointStruct(id=r["chunk_id"], vector=vecs[b + j].tolist(),
                        payload={k: v for k, v in r.items() if v is not None})
            for j, r in enumerate(shard[b:b + UPSERT_BATCH])
        ]
        qdrant_client.upsert(collection_name=COLLECTION_NAME, points=pts)
    done.add(s)
    json.dump(sorted(done), open(PROGRESS_PATH, "w"))   # checkpoint after each shard
    pbar.update(hi - lo)
    del vecs, shard
    gc.collect()
pbar.close()

# --- All points in: enable indexing -> HNSW builds once --------------------
print()
print("All shards uploaded. Enabling HNSW indexing (builds in background)...")
qdrant_client.update_collection(
    collection_name=COLLECTION_NAME,
    optimizers_config=OptimizersConfigDiff(indexing_threshold=100),
)
# Search already works before this finishes (brute-force on unindexed segments);
# poll until the optimizer reports GREEN so queries are fast.
deadline = time.time() + 7200          # 723k vectors take well over 30 min to index
_t0 = time.time()
while time.time() < deadline:
    info = qdrant_client.get_collection(COLLECTION_NAME)
    if str(info.status).split(".")[-1].lower() == "green":
        print(f"HNSW built in {(time.time()-_t0)/60:.1f} min")
        break
    print(f"  indexing... {info.indexed_vectors_count:,}/{info.points_count:,} "
          f"({(time.time()-_t0)/60:.1f} min elapsed)")
    time.sleep(30)
else:
    print("[WARN] HNSW still building after 2h. Search WORKS but is slower on the\n"
          "       unindexed segments. Re-check get_collection(...).status later.")

info = qdrant_client.get_collection(COLLECTION_NAME)
print()
print(f"Collection '{COLLECTION_NAME}' ready")
print(f"  points:        {info.points_count:,}")
print(f"  indexed vecs:  {info.indexed_vectors_count:,} / {info.points_count:,}")
print(f"  status:        {info.status}")
print(f"  storage:       {QDRANT_PATH}")
print("  -> copy this folder (+ shard_progress.json) to Drive to persist across sessions!")


embedding cache: /content/drive/MyDrive/rag_emb_cache_bge_m3_full  (12 shards already embedded)
Fresh collection 'agri_knowledge' created (HNSW indexing deferred)


Indexing corpus:   0%|          | 0/723439 [00:00<?, ?chunk/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Batches:   0%|          | 0/54 [00:00<?, ?it/s]


All shards uploaded. Enabling HNSW indexing (builds in background)...
  indexing... 0/723,439 (0.0 min elapsed)
  indexing... 0/723,439 (0.5 min elapsed)
  indexing... 0/723,439 (1.0 min elapsed)
  indexing... 0/723,439 (1.5 min elapsed)
  indexing... 0/723,439 (2.0 min elapsed)
  indexing... 0/723,439 (2.5 min elapsed)
  indexing... 0/723,439 (3.0 min elapsed)
  indexing... 128,160/723,439 (3.5 min elapsed)
  indexing... 128,160/723,439 (4.0 min elapsed)
  indexing... 128,160/723,439 (4.5 min elapsed)
  indexing... 128,160/723,439 (5.0 min elapsed)
  indexing... 128,160/723,439 (5.5 min elapsed)
  indexing... 256,320/723,439 (6.0 min elapsed)
  indexing... 256,320/723,439 (6.5 min elapsed)
  indexing... 256,320/723,439 (7.0 min elapsed)
  indexing... 256,320/723,439 (7.5 min elapsed)
  indexing... 256,320/723,439 (8.0 min elapsed)
  indexing... 384,480/723,439 (8.5 min elapsed)
  indexing... 384,480/723,439 (9.0 min elapsed)
  indexing... 384,480/723,439 (9.5 min elapsed)
  indexing.

## 4. LLM Tool — `search_agri_knowledge`

The single retrieval entrypoint the agentic LLM registers (M3 report §10.7, Appendix D):
JSON-in / JSON-out, **never raises** (errors return `tier: "error"`), every hit cited
(PDF -> file + pages + section; KCC -> record + district + year).

**Coexistence at query time — weighted per-source fusion, not one flat search.** The KCC
corpus outnumbers PDF ~100:1; a flat top-k would drown scheme/policy content. The tool runs
one filtered sub-query per corpus and fuses with intent weights (`policy` -> PDF 2.0/KCC 0.5,
`field_practice` -> KCC 2.0/PDF 0.5, `general` -> 1.0/1.0). Callers can also pin
`source_type` explicitly. The Milestone-1 relevance tier (grounded / fallback / abstain)
is decided on the best **raw** cosine score, never the fused one.

In [12]:
# Filter primitives for the tool (self-contained -- the build cell no longer
# exports these into the global scope).
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range

RAG_TOOL_SPEC = {
    "name": "search_agri_knowledge",
    "description": (
        "Search the UP agricultural knowledge base: government scheme guidelines, "
        "pest/disease advisories and district contingency plans (PDF corpus) plus "
        "Kisan Call Centre farmer Q&A with expert answers (KCC corpus). Returns "
        "cited chunks with a relevance tier. Query may be English or Hindi."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query":       {"type": "string", "description": "The farmer's question, English or Hindi"},
            "top_k":       {"type": "integer", "default": 5, "minimum": 1, "maximum": 20},
            "intent":      {"type": "string", "enum": ["policy", "field_practice", "general"],
                            "description": "Weights the PDF-vs-KCC fusion; default general"},
            "source_type": {"type": "string", "enum": ["pdf", "kcc"],
                            "description": "Pin one corpus; omit for weighted search over both"},
            "doc_category": {"type": "string", "enum": ["scheme_eligibility", "crop_advisory",
                                                        "contingency_plan", "policy_guideline"],
                             "description": "PDF corpus only"},
            "query_type":  {"type": "string", "description": "KCC only, e.g. 'Plant Protection'"},
            "crop":        {"type": "string", "description": "Canonical crop name (rice, wheat, ...)"},
            "district":    {"type": "string", "description": "Canonical UP district name"},
            "season":      {"type": "string", "enum": ["Rabi", "Kharif", "Zaid"], "description": "KCC only"},
            "language":    {"type": "string", "enum": ["en", "hi", "mixed"]},
            "year_from":   {"type": "integer", "description": "Only content from this year onward"},
            "only_tables": {"type": "boolean", "description": "PDF dosage/scheme tables only"},
        },
        "required": ["query"],
    },
}


def _citation(p):
    if p.get("source_type") == "pdf":
        # `district` is populated for ACP contingency plans (named after the
        # district). It was missing from this branch, so a district-filtered
        # search returned correct PDF hits whose citation reported district=None
        # -- eval check A4 then flagged them as violations. Retrieval was fine;
        # the citation was lossy.
        return {"corpus": "pdf", "file": p.get("filename"),
                "pages": [p.get("page_start"), p.get("page_end")],
                "section": p.get("heading_hierarchy") or None,
                "doc_category": p.get("doc_category"),
                "district": p.get("district"), "year": p.get("year")}
    return {"corpus": "kcc", "record": "KCC Q&A", "crop": p.get("crop"),
            "district": p.get("district"), "season": p.get("season"),
            "query_type": p.get("query_type"), "year": p.get("year")}


def search_agri_knowledge(query, top_k=TOP_K_DEFAULT, intent="general", source_type=None,
                          doc_category=None, query_type=None, crop=None, district=None,
                          season=None, language=None, year_from=None, only_tables=None):
    """LLM tool entrypoint: JSON-in/JSON-out, never raises, always cites."""
    weights = FUSION_WEIGHTS.get(intent, FUSION_WEIGHTS["general"])

    def sub_search(stype):
        must = [FieldCondition(key="source_type", match=MatchValue(value=stype))]
        if doc_category: must.append(FieldCondition(key="doc_category", match=MatchValue(value=doc_category)))
        if query_type:   must.append(FieldCondition(key="query_type", match=MatchValue(value=query_type)))
        if crop:         must.append(FieldCondition(key="crop", match=MatchValue(value=canon_crop(crop))))
        if district:     must.append(FieldCondition(key="district", match=MatchValue(value=canon_district(district))))
        if season:       must.append(FieldCondition(key="season", match=MatchValue(value=season)))
        if language:     must.append(FieldCondition(key="language", match=MatchValue(value=language)))
        if year_from:    must.append(FieldCondition(key="year", range=Range(gte=year_from)))
        if only_tables:  must.append(FieldCondition(key="has_table", match=MatchValue(value=True)))
        return qdrant_client.query_points(
            collection_name=COLLECTION_NAME,
            query=qvec,
            query_filter=Filter(must=must),
            limit=top_k,
            with_payload=True,
        ).points

    try:
        # e5 is asymmetric: the QUERY side takes "query: ", never "passage: ".
        qvec = embed_model.encode(embed_query(query), normalize_embeddings=True).tolist()
        sources = [source_type] if source_type else ["pdf", "kcc"]
        hits = []
        for stype in sources:
            for h in sub_search(stype):
                hits.append({
                    "raw_score": round(float(h.score), 4),
                    "fused_score": round(float(h.score) * weights.get(stype, 1.0), 4),
                    "text": h.payload.get("text", ""),
                    "source_type": stype,
                    "has_table": bool(h.payload.get("has_table", False)),
                    "chunk_id": h.payload.get("chunk_id"),
                    "citation": _citation(h.payload),
                })
        hits.sort(key=lambda x: x["fused_score"], reverse=True)
        hits = hits[:top_k]
    except Exception as e:
        return {"query": query, "tier": "error", "top_score": 0.0,
                "results": [], "error": str(e)}

    best_raw = max((h["raw_score"] for h in hits), default=0.0)  # tier on RAW cosine
    tier = ("grounded" if best_raw >= TIER_GROUNDED
            else "fallback_with_disclaimer" if best_raw >= TIER_FALLBACK
            else "abstain_out_of_scope")
    return {"query": query, "intent": intent, "tier": tier,
            "top_score": round(best_raw, 4), "results": hits}


print("Tool ready: search_agri_knowledge(...)")
print(json.dumps(RAG_TOOL_SPEC, indent=2)[:600] + " ...")

# Smoke call
_s = search_agri_knowledge("interest subvention on crop loans", top_k=3, intent="policy")
print(f"\nSmoke: tier={_s['tier']} top={_s['top_score']} "
      f"sources={[r['source_type'] for r in _s['results']]}")

Tool ready: search_agri_knowledge(...)
{
  "name": "search_agri_knowledge",
  "description": "Search the UP agricultural knowledge base: government scheme guidelines, pest/disease advisories and district contingency plans (PDF corpus) plus Kisan Call Centre farmer Q&A with expert answers (KCC corpus). Returns cited chunks with a relevance tier. Query may be English or Hindi.",
  "parameters": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "The farmer's question, English or Hindi"
      },
      "top_k": {
        "type": "integer",
        "default": 5,
        "minimum ...

Smoke: tier=grounded top=0.7546 sources=['pdf', 'pdf', 'pdf']


## 5. Evaluation — Farmer Queries Through the Tool

Every check below goes through `search_agri_knowledge` exactly as the LLM will call it.

- **A. Filter correctness** — every returned hit must satisfy its metadata filter
- **B. Bilingual retrieval** — the same intent in English / Devanagari / Hinglish should
  agree on what it retrieves (M1 objective O3 is code-mixed retrieval)
- **C. Corpus routing** — `policy` intent should surface PDF scheme content at the top;
  `field_practice` should surface KCC expert answers (validates the fusion weights)
- **D. Domain separation + tier recalibration** — off-domain queries (car repair, chess,
  stock market, football, Python) must score clearly below in-domain ones, or tier-based
  abstention cannot work. Because thresholds are **model-specific**, `TIER_FALLBACK` and
  `TIER_GROUNDED` are *derived here from the live index* (8 in-domain vs 6 off-domain
  probes) and applied to the session — the printed pair goes back into the config cell.
  Carrying over another model's thresholds is what made the old 0.85/0.65 meaningless.
- **E. Latency** — per-call wall time (embed + filtered HNSW + fusion)

Output: PASS/WARN/FAIL table + `rag_eval_report.json` beside the Qdrant store.

In [13]:
report, timings = [], []

def timed_call(**kw):
    t0 = time.time()
    out = search_agri_knowledge(**kw)
    ms = (time.time() - t0) * 1000
    timings.append(ms)
    out["_ms"] = round(ms, 1)
    return out

def check(label, out, predicate):
    hits = out["results"]
    if out["tier"] == "error":
        report.append((label, "FAIL", out.get("error", "")[:80]))
    elif not hits:
        report.append((label, "WARN", "0 results — over-restrictive filter or corpus gap"))
    else:
        bad = [h for h in hits if not predicate(h)]
        report.append((label, "PASS" if not bad else "FAIL",
                       f"{len(hits)} hits, {len(bad)} violate, top={out['top_score']:.3f}, "
                       f"tier={out['tier']}, {out['_ms']:.0f}ms"))

# --- A. Filter correctness ---------------------------------------------------
check("A1 pdf-only + scheme category",
      timed_call(query="who is eligible for interest subvention on crop loans",
                 source_type="pdf", doc_category="scheme_eligibility"),
      lambda h: h["source_type"] == "pdf" and h["citation"]["doc_category"] == "scheme_eligibility")
check("A2 kcc-only + crop=rice",
      timed_call(query="fertilizer dose for paddy nursery", source_type="kcc", crop="rice"),
      lambda h: h["source_type"] == "kcc" and h["citation"]["crop"] == "rice")
check("A3 dosage tables only (pdf)",
      timed_call(query="approved fungicides and dosage for rice blast", only_tables=True),
      lambda h: h["has_table"])
check("A4 district filter (canonicalized: allahabad->prayagraj)",
      timed_call(query="crop advice for my district", district="allahabad"),
      lambda h: h["citation"].get("district") == "prayagraj")
check("A5 season filter (kcc Rabi)",
      timed_call(query="wheat sowing time", season="Rabi"),
      lambda h: h["citation"].get("season") == "Rabi")
check("A6 year_from=2020",
      timed_call(query="latest pest advisory", year_from=2020),
      lambda h: (h["citation"].get("year") or 0) >= 2020)

# --- B. Bilingual / code-mixed consistency -----------------------------------
INTENTS = [
    ("wheat irrigation", ["when should wheat be irrigated and how many times",
                          "गेहूं की सिंचाई कब और कितनी बार करें",
                          "gehu me sinchai kab karni chahiye"]),
    ("rice blast",       ["treatment for blast disease in paddy",
                          "धान में ब्लास्ट रोग का उपचार",
                          "dhan me blast rog ka ilaj"]),
]
for label, forms in INTENTS:
    got = []
    for q in forms:
        out = timed_call(query=q, top_k=5)
        got.append({(r["citation"].get("file") or r["chunk_id"]) for r in out["results"]})
    j_en_hi = len(got[0] & got[1]) / max(len(got[0] | got[1]), 1)
    hing = len(got[0] & got[2])
    report.append((f"B  bilingual [{label}]", "PASS" if j_en_hi >= 0.2 else "WARN",
                   f"EN~HI jaccard={j_en_hi:.2f}, hinglish overlap with EN={hing}/5"))

# --- C. Corpus routing via fusion weights ------------------------------------
pol = timed_call(query="pm kisan samman nidhi eligibility and benefits", intent="policy")
n_pdf_top = sum(1 for r in pol["results"][:3] if r["source_type"] == "pdf")
report.append(("C1 policy intent -> pdf-heavy top-3",
               "PASS" if n_pdf_top >= 2 else "WARN",
               f"{n_pdf_top}/3 pdf in top-3, tier={pol['tier']}"))
fld = timed_call(query="yellowing in onion nursery what spray to use", intent="field_practice")
n_kcc_top = sum(1 for r in fld["results"][:3] if r["source_type"] == "kcc")
report.append(("C2 field intent -> kcc-heavy top-3",
               "PASS" if n_kcc_top >= 2 else "WARN",
               f"{n_kcc_top}/3 kcc in top-3, tier={fld['tier']}"))

# --- D. Domain separation + TIER RECALIBRATION -------------------------------
# Thresholds are model-specific, so they are DERIVED here from the live index
# rather than trusted from config. More probes than the old 3-vs-3 so the
# percentiles mean something.
in_dom = ["recommended fertilizer schedule for wheat in up",
          "fall army worm control in maize",
          "गेहूं में पीला रतुआ की रोकथाम",
          "how much urea in wheat at tillering stage",
          "dhan ki nursery me pili patti ka ilaj",
          "pm kisan samman nidhi eligibility",
          "आम में भुनगा कीट का उपचार",
          "subsidy for drip irrigation in uttar pradesh"]
off_dom = ["how do I repair my car engine", "best chess opening strategy",
           "शेयर बाजार में निवेश कैसे करें", "who won the football world cup",
           "how to write a python for loop", "मुझे नई कार खरीदनी है"]
in_s  = [timed_call(query=q, top_k=1)["top_score"] for q in in_dom]
off_s = [timed_call(query=q, top_k=1)["top_score"] for q in off_dom]
margin = min(in_s) - max(off_s)
report.append(("D1 domain separation", "PASS" if margin > 0.03 else "WARN",
               f"in-domain min={min(in_s):.3f}, off-domain max={max(off_s):.3f}, margin={margin:+.3f}"))

# Abstain floor: put the boundary between the two populations. When they overlap
# (margin <= 0) no clean cut exists -- sit just above the off-domain 90th pct and
# say so, because tier-based abstention is then unreliable by construction.
_p = lambda xs, q: float(np.percentile(xs, q))
if margin > 0:
    NEW_FALLBACK = round((min(in_s) + max(off_s)) / 2, 3)
else:
    NEW_FALLBACK = round(_p(off_s, 90) + 0.01, 3)
# GROUNDED at the in-domain p25, NOT p50. The median is self-defeating: half of
# all genuine farm queries score below it by definition, so correct retrievals get
# downgraded to "fallback" and served with a disclaimer. Measured on the 30k run
# with p50=0.868: PM-KISAN (0.866, correct operational guidelines) and tomato leaf
# curl (0.862, correct imidacloprid dose) were both downgraded. p25 encodes the
# intended meaning -- "3 in 4 genuine queries should clear the confidence bar".
# The +0.02 floor below keeps GROUNDED clear of FALLBACK when the percentile is
# noisy (it is estimated from only 8 probes).
NEW_GROUNDED = round(max(_p(in_s, 25), NEW_FALLBACK + 0.02), 3)

report.append(("D2 tier recalibration", "INFO" if margin > 0 else "WARN",
               f"in-domain p50={_p(in_s,50):.3f} min={min(in_s):.3f} | "
               f"off-domain p90={_p(off_s,90):.3f} max={max(off_s):.3f} -> "
               f"TIER_FALLBACK={NEW_FALLBACK} TIER_GROUNDED={NEW_GROUNDED}"
               + ("" if margin > 0 else "  [populations OVERLAP — abstention unreliable]")))

# Apply to THIS session so every check below (and the inspector cell) uses the
# measured values; the printed line goes back into the config cell to persist.
TIER_FALLBACK, TIER_GROUNDED = NEW_FALLBACK, NEW_GROUNDED
print(f"[calibrated] TIER_FALLBACK = {TIER_FALLBACK}   TIER_GROUNDED = {TIER_GROUNDED}")
print("             ^ paste these into the config cell to make them permanent\n")

# --- Report ------------------------------------------------------------------
print("=" * 92)
print("RAG EVALUATION — farmer queries through search_agri_knowledge")
print("=" * 92)
counts = Counter(s for _, s, _ in report)
for label, status, detail in report:
    print(f"  [{status:4}] {label:44} {detail}")
print("-" * 92)
print(f"  {counts['PASS']} pass / {counts['FAIL']} fail / {counts['WARN']} warn / "
      f"{counts['INFO']} info | latency p50={np.percentile(timings,50):.0f}ms "
      f"p95={np.percentile(timings,95):.0f}ms over {len(timings)} calls")

eval_path = os.path.join(RAG_DB_DIR, "rag_eval_report.json")
with open(eval_path, "w", encoding="utf-8") as f:
    json.dump({"checks": [dict(zip(("label","status","detail"), r)) for r in report],
               "latency_ms": {"p50": float(np.percentile(timings, 50)),
                              "p95": float(np.percentile(timings, 95))},
               "collection": COLLECTION_NAME, "model": EMBED_MODEL_NAME,
               "tiers": {"fallback": TIER_FALLBACK, "grounded": TIER_GROUNDED},
               "score_dist": {"in_domain": in_s, "off_domain": off_s},
               "n_chunks": len(unified)}, f, indent=2, ensure_ascii=False)
print(f"  saved: {eval_path}")

# --- Sample tool response (what the LLM will actually receive) ---------------
print("\nSAMPLE — search_agri_knowledge('paddy blast dose', only_tables=True):")
print(json.dumps(timed_call(query="paddy blast treatment dose", top_k=2, only_tables=True),
                 indent=2, ensure_ascii=False)[:1600] + " ...")

[calibrated] TIER_FALLBACK = 0.56   TIER_GROUNDED = 0.66
             ^ paste these into the config cell to make them permanent

RAG EVALUATION — farmer queries through search_agri_knowledge
  [PASS] A1 pdf-only + scheme category                5 hits, 0 violate, top=0.681, tier=grounded, 210ms
  [PASS] A2 kcc-only + crop=rice                      5 hits, 0 violate, top=0.801, tier=grounded, 586ms
  [PASS] A3 dosage tables only (pdf)                  5 hits, 0 violate, top=0.788, tier=grounded, 464ms
  [PASS] A4 district filter (canonicalized: allahabad->prayagraj) 5 hits, 0 violate, top=0.638, tier=fallback_with_disclaimer, 996ms
  [PASS] A5 season filter (kcc Rabi)                  5 hits, 0 violate, top=0.811, tier=grounded, 492ms
  [PASS] A6 year_from=2020                            5 hits, 0 violate, top=0.625, tier=fallback_with_disclaimer, 982ms
  [WARN] B  bilingual [wheat irrigation]              EN~HI jaccard=0.00, hinglish overlap with EN=0/5
  [WARN] B  bilingual [rice blas

In [14]:
# ---------------------------------------------------------------------------
# 6. Retrieval Inspector — the ACTUAL chunks behind each kind of query
# ---------------------------------------------------------------------------
# Section 5 reports PASS/WARN/FAIL but never shows WHAT came back. This cell
# prints the retrieved text itself so retrieval quality can be judged by eye,
# across the full query space the system has to handle:
#   A  policy / scheme       -> should route PDF-heavy   (FUSION_WEIGHTS)
#   B  field practice / how-to -> should route KCC-heavy (FUSION_WEIGHTS)
#   C  disease diagnosis     -> the vision-module handoff (M3 report §3)
#   D  metadata-filtered     -> filters + district canonicalization (M3 §9.5)
#   E  edge cases            -> off-domain abstention, vague, compound queries
# A/B/C are each run in English / Devanagari / Hinglish where relevant —
# code-mixed retrieval is the specific reason MuRIL was chosen (M3 §5.2).
# ---------------------------------------------------------------------------

SNIPPET_CHARS = 300      # chars of chunk text to print; set to None for the full chunk
TOP_N_SHOW    = 3        # how many hits to display per query


def _snip(text, n=SNIPPET_CHARS):
    """Collapse whitespace (KCC chunks are multi-line Q&A) and truncate."""
    t = re.sub(r"\s+", " ", str(text or "")).strip()
    if n is None or len(t) <= n:
        return t
    return t[:n].rstrip() + " ..."


def _fmt_citation(c):
    """One-line provenance; PDF and KCC carry different fields."""
    if c.get("corpus") == "pdf":
        bits = [str(c.get("file") or "?")]
        pages = c.get("pages") or [None, None]
        ps, pe = (pages + [None, None])[:2]
        if ps is not None:
            bits.append(f"p.{ps}" if pe in (None, ps) else f"pp.{ps}-{pe}")
        for k in ("doc_category", "year"):
            if c.get(k):
                bits.append(str(c[k]))
        if c.get("section"):
            bits.append(f"sec: {_snip(c['section'], 60)}")
        return " | ".join(bits)
    bits = ["KCC Q&A"]
    for k in ("crop", "district", "season", "query_type", "year"):
        if c.get(k):
            bits.append(str(c[k]))
    return " | ".join(bits)


# --- The query battery: (category, label, kwargs passed to the tool) ---------
INSPECT_QUERIES = [
    # A. Policy / scheme — fusion should push PDF to the top
    ("A. POLICY / SCHEME  (expect PDF-heavy)", "EN  · loan interest subvention",
     dict(query="who is eligible for interest subvention on crop loans", intent="policy")),
    ("A. POLICY / SCHEME  (expect PDF-heavy)", "EN  · PM-KISAN benefits",
     dict(query="pm kisan samman nidhi eligibility and benefits", intent="policy")),
    ("A. POLICY / SCHEME  (expect PDF-heavy)", "HI  · fasal bima yojana",
     dict(query="प्रधानमंत्री फसल बीमा योजना के लाभ और पात्रता क्या है", intent="policy")),

    # B. Field practice / how-to — fusion should push KCC to the top
    ("B. FIELD PRACTICE  (expect KCC-heavy)", "EN  · urea dose in wheat",
     dict(query="how much urea should be applied in wheat at tillering stage",
          intent="field_practice")),
    ("B. FIELD PRACTICE  (expect KCC-heavy)", "HI  · weed control in wheat",
     dict(query="गेहूं में खरपतवार नियंत्रण के लिए कौन सी दवा डालें",
          intent="field_practice")),
    ("B. FIELD PRACTICE  (expect KCC-heavy)", "HING· paddy nursery yellowing",
     dict(query="dhan ki nursery me pili patti ho rahi hai kya karein",
          intent="field_practice")),

    # C. Disease diagnosis — what the vision module hands off to retrieval
    ("C. DISEASE / DIAGNOSIS", "EN  · brown spots on rice",
     dict(query="brown spots on rice leaves what disease is this and how to control")),
    ("C. DISEASE / DIAGNOSIS", "HI  · tomato leaf curl",
     dict(query="टमाटर के पौधे में पत्तियां मुड़ रही हैं क्या करें")),
    ("C. DISEASE / DIAGNOSIS", "HING· wheat yellow rust",
     dict(query="gehu me pila ratua lag gaya hai konsi dawa daale")),

    # D. Metadata-filtered retrieval (M3 §9.5)
    ("D. FILTERED RETRIEVAL", "dosage TABLES only (pdf)",
     dict(query="approved fungicides and dosage for rice blast", only_tables=True)),
    ("D. FILTERED RETRIEVAL", "district canon: allahabad -> prayagraj",
     dict(query="contingency plan for delayed monsoon", district="allahabad")),
    ("D. FILTERED RETRIEVAL", "crop=wheat + season=Rabi (kcc)",
     dict(query="sowing time and seed rate", crop="wheat", season="Rabi")),
    ("D. FILTERED RETRIEVAL", "pinned pdf + scheme_eligibility",
     dict(query="subsidy application process and required documents",
          source_type="pdf", doc_category="scheme_eligibility")),

    # E. Edge cases — abstention, ambiguity, compound intent
    ("E. EDGE CASES", "OFF-DOMAIN en (expect abstain)",
     dict(query="how do I repair my motorcycle engine")),
    ("E. EDGE CASES", "OFF-DOMAIN hi (expect abstain)",
     dict(query="शेयर बाजार में निवेश कैसे करें")),
    ("E. EDGE CASES", "VAGUE (tests §9.7 re-rank case)",
     dict(query="meri fasal kharab ho rahi hai kya karu")),
    ("E. EDGE CASES", "COMPOUND (advice + market)",
     dict(query="wheat me kitna urea dalna hai aur mandi rate kya chal raha hai")),
]

W = 100
print("=" * W)
print("RETRIEVAL INSPECTOR — what each kind of query actually pulls back")
print("=" * W)
if KCC_MAX_CHUNKS:
    print(f"NOTE: DEV subset active (KCC<={KCC_MAX_CHUNKS:,}). Narrow filters (district,")
    print("      season, year) may return few or 0 hits simply because sampling thinned")
    print("      that slice — that is expected here, not a retrieval failure.")
    print("-" * W)

inspection, summary, _cat = [], [], None
for cat, label, kw in INSPECT_QUERIES:
    if cat != _cat:
        _cat = cat
        print(f"\n{'#' * W}\n## {cat}\n{'#' * W}")

    t0 = time.time()
    out = search_agri_knowledge(**{**kw, "top_k": max(TOP_N_SHOW, TOP_K_DEFAULT)})
    ms = (time.time() - t0) * 1000

    params = ", ".join(f"{k}={v!r}" for k, v in kw.items() if k != "query") or "—"
    hits = out.get("results", [])
    n_pdf_h = sum(1 for h in hits[:TOP_N_SHOW] if h["source_type"] == "pdf")
    n_kcc_h = sum(1 for h in hits[:TOP_N_SHOW] if h["source_type"] == "kcc")

    print(f"\n{'-' * W}")
    print(f"[{label}]")
    print(f"  Q      : {kw['query']}")
    print(f"  params : {params}")
    if out.get("tier") == "error":
        print(f"  RESULT : ERROR — {out.get('error', '')[:120]}")
        summary.append((label, "error", 0.0, 0, 0, ms))
        continue
    print(f"  result : tier={out['tier']}  top_raw={out['top_score']:.3f}  "
          f"{ms:.0f}ms  |  top-{TOP_N_SHOW}: {n_pdf_h} pdf / {n_kcc_h} kcc")

    if not hits:
        print("  (no hits — over-restrictive filter, or this slice is absent from the corpus)")
    for i, h in enumerate(hits[:TOP_N_SHOW], 1):
        tbl = " [TABLE]" if h.get("has_table") else ""
        print(f"\n   {i}. raw={h['raw_score']:.3f}  fused={h['fused_score']:.3f}  "
              f"<{h['source_type']}>{tbl}")
        print(f"      {_fmt_citation(h['citation'])}")
        print(f"      \"{_snip(h['text'])}\"")

    summary.append((label, out["tier"], out["top_score"], n_pdf_h, n_kcc_h, ms))
    inspection.append({"category": cat, "label": label, "params": kw,
                       "tier": out["tier"], "top_score": out["top_score"],
                       "latency_ms": round(ms, 1),
                       "hits": [{"raw_score": h["raw_score"], "fused_score": h["fused_score"],
                                 "source_type": h["source_type"], "citation": h["citation"],
                                 "text": h["text"]} for h in hits[:TOP_N_SHOW]]})

# --- Compact routing/scoring overview ---------------------------------------
print(f"\n{'=' * W}")
print("SUMMARY — routing and score distribution at a glance")
print("=" * W)
print(f"  {'query':<42} {'tier':<26} {'top':>6} {'pdf':>4} {'kcc':>4} {'ms':>6}")
print("  " + "-" * (W - 4))
for label, tier, top, npdf, nkcc, ms in summary:
    print(f"  {label:<42} {tier:<26} {top:>6.3f} {npdf:>4} {nkcc:>4} {ms:>6.0f}")

_in_scores = [s for lbl, t, s, *_ in summary if not lbl.startswith("OFF-DOMAIN") and s > 0]
_off_scores = [s for lbl, t, s, *_ in summary if lbl.startswith("OFF-DOMAIN")]
if _in_scores and _off_scores:
    print("  " + "-" * (W - 4))
    print(f"  in-domain min={min(_in_scores):.3f} | off-domain max={max(_off_scores):.3f} "
          f"| separation={min(_in_scores) - max(_off_scores):+.3f}")
    print(f"  current tiers: abstain<{TIER_FALLBACK} <=fallback< {TIER_GROUNDED} <=grounded")

insp_path = os.path.join(RAG_DB_DIR, "retrieval_inspection.json")
with open(insp_path, "w", encoding="utf-8") as f:
    json.dump({"scale": "full" if KCC_MAX_CHUNKS is None else f"dev_{KCC_MAX_CHUNKS}",
               "n_chunks": len(unified), "model": EMBED_MODEL_NAME,
               "queries": inspection}, f, indent=2, ensure_ascii=False)
print(f"\n  full retrieved text saved -> {insp_path}")


RETRIEVAL INSPECTOR — what each kind of query actually pulls back

####################################################################################################
## A. POLICY / SCHEME  (expect PDF-heavy)
####################################################################################################

----------------------------------------------------------------------------------------------------
[EN  · loan interest subvention]
  Q      : who is eligible for interest subvention on crop loans
  params : intent='policy'
  result : tier=grounded  top_raw=0.681  153ms  |  top-3: 3 pdf / 0 kcc

   1. raw=0.681  fused=1.362  <pdf>
      MODIFIED_INTEREST_SUBVENTION_SCHEME.pdf | p.1 | scheme_eligibility | 2026 | sec: **MODIFIED INTEREST SUBVENTION SCHEME FOR SHORT TERM LOANS F ...
      "* * MODIFIED INTEREST SUBVENTION SCHEME FOR SHORT TERM LOANS FOR AGRIC - **2)** An additional interest subvention of 3% per annum will be provided to such of those farmers repaying in time, i.e.

In [ ]:
# ---------------------------------------------------------------------------
# 7. SCORECARD — one comparable summary per model
# ---------------------------------------------------------------------------
# Writes a standardized JSON into a SHARED directory so the e5-base / e5-large /
# bge-m3 runs can be diffed directly, and prints the same block to the screen.
# Every value is read defensively: if an upstream cell was skipped the scorecard
# still writes rather than dying at the end of a long run.
import os, json, numpy as np

_g = globals()
def _get(name, default=None):
    v = _g.get(name, default)
    return default if v is None else v

CMP_DIR = "/content/model_comparison" if os.path.isdir("/content") else "../data/final/model_comparison"
os.makedirs(CMP_DIR, exist_ok=True)

_counts = {}
for _, _st, _ in _get("report", []):
    _counts[_st] = _counts.get(_st, 0) + 1

_tiers = {}
for _row in _get("summary", []):
    _tiers[_row[1]] = _tiers.get(_row[1], 0) + 1

_t = _get("timings", [])
scorecard = {
    "model_tag":      MODEL_TAG,
    "model":          EMBED_MODEL_NAME,
    "dim":            _get("EMBED_DIM", 0),
    "prefixes":       {"query": QUERY_PREFIX, "doc": DOC_PREFIX},
    "device":         _get("device"),
    "corpus": {
        "total_chunks": len(_get("unified", [])),
        "pdf":          _get("n_pdf", 0),
        "kcc":          len(_get("unified", [])) - _get("n_pdf", 0),
        "kcc_max":      _get("KCC_MAX_CHUNKS", 0),
    },
    "load_gate": {
        "ranking_correct": _get("_correct", 0),
        "ranking_total":   len(_get("GATE_QUERIES", [])),
        "mean_gold_margin": round(float(np.mean(_get("_margins", [0]))), 4),
        "unrelated_doc_cosine_median": round(float(np.median(_get("_off", [0]))), 4),
    },
    "eval": {
        "pass": _counts.get("PASS", 0), "fail": _counts.get("FAIL", 0),
        "warn": _counts.get("WARN", 0), "info": _counts.get("INFO", 0),
        "checks": [dict(zip(("label", "status", "detail"), r)) for r in _get("report", [])],
    },
    "separation": {
        "in_domain_min":  round(min(_get("in_s", [0])), 4),
        "off_domain_max": round(max(_get("off_s", [0])), 4),
        "margin":         round(min(_get("in_s", [0])) - max(_get("off_s", [0])), 4),
        "in_domain_scores":  [round(x, 4) for x in _get("in_s", [])],
        "off_domain_scores": [round(x, 4) for x in _get("off_s", [])],
    },
    "tiers_calibrated": {"fallback": TIER_FALLBACK, "grounded": TIER_GROUNDED},
    "tier_distribution_on_inspector": _tiers,
    "latency_ms": {"p50": round(float(np.percentile(_t, 50)), 1) if _t else None,
                   "p95": round(float(np.percentile(_t, 95)), 1) if _t else None},
    "note": "local Qdrant = exact brute-force; latency is NOT the M3 §12.3 server-mode number",
}

_path = os.path.join(CMP_DIR, f"scorecard_{MODEL_TAG}.json")
with open(_path, "w", encoding="utf-8") as f:
    json.dump(scorecard, f, indent=2, ensure_ascii=False)

W = 78
print("=" * W)
print(f"SCORECARD — {EMBED_MODEL_NAME}")
print("=" * W)
g = scorecard["load_gate"]; s = scorecard["separation"]; e = scorecard["eval"]
print(f"  dim / batch        : {scorecard['dim']} / {EMBED_BATCH_SIZE}")
print(f"  prefixes           : query={QUERY_PREFIX!r}  doc={DOC_PREFIX!r}")
print(f"  corpus             : {scorecard['corpus']['total_chunks']:,} chunks "
      f"(pdf={scorecard['corpus']['pdf']:,}, kcc={scorecard['corpus']['kcc']:,})")
print(f"  GATE ranking       : {g['ranking_correct']}/{g['ranking_total']}"
      f"   mean margin {g['mean_gold_margin']:+.4f}"
      f"   unrelated-doc cos {g['unrelated_doc_cosine_median']:.3f}")
print(f"  EVAL               : {e['pass']} pass / {e['fail']} fail / {e['warn']} warn")
print(f"  SEPARATION         : in-min {s['in_domain_min']:.3f}  off-max {s['off_domain_max']:.3f}"
      f"  margin {s['margin']:+.4f}   <-- higher is better")
print(f"  tiers (calibrated) : fallback {TIER_FALLBACK}  grounded {TIER_GROUNDED}")
print(f"  inspector tiers    : {scorecard['tier_distribution_on_inspector']}")
print(f"  latency            : p50 {scorecard['latency_ms']['p50']}ms  p95 {scorecard['latency_ms']['p95']}ms")
print("-" * W)
print("  KEY COMPARISON METRICS (higher = better, except latency):")
print(f"    gate margin {g['mean_gold_margin']:+.4f} | separation {s['margin']:+.4f} | "
      f"eval pass {e['pass']}/{e['pass']+e['fail']+e['warn']}")
print(f"\n  saved -> {_path}")
print("  Run all three notebooks, then compare the scorecard_*.json files in this folder.")


In [16]:
# ---------------------------------------------------------------------------
# SUPERSEDED — this was the LOCAL-mode export (zip the qdrant_db folder).
# ---------------------------------------------------------------------------
# It no longer applies now that the notebook runs Qdrant in SERVER mode:
#   * it pointed at /content/rag_outputs_bge_m3_dev_100k, which the full
#     production build does not create (RAG_DB_DIR is now ..._full)
#   * it called qdrant_client.close(), which would drop the server connection
#     that every cell after it needs
#   * zipping a live storage folder can capture half-written segments; the
#     server's own snapshot API produces a consistent file instead
# Use section 8 (EXPORT) at the end of the notebook.
print('superseded — use section 8 (EXPORT) below; nothing to do here')


superseded — use section 8 (EXPORT) below; nothing to do here


In [ ]:
# (merged into section 8 EXPORT above — kept empty to avoid running twice)


## 8–9. Persist and Restore

A full build is hours of T4 time. Two layers protect it:

| Artifact | What it saves you | Where |
|---|---|---|
| **Embedding cache** (`EMB_CACHE_DIR`) | the GPU stage — rebuild the index without re-embedding | Drive, written per shard during the build |
| **Qdrant snapshot** (section 8) | the whole built index, HNSW included | Drive, one file |

The snapshot is taken through Qdrant's own API, not by zipping the storage folder — zipping a live directory can capture half-written segments. The **manifest** alongside it records the model id, prefixes, dimension, calibrated tiers and fusion weights: restoring vectors alone is not enough, since a different prefix or threshold silently changes every answer.

In [ ]:
import os, json, time, shutil, requests

DEST = "/content/drive/MyDrive/rag_production_bge_m3"
os.makedirs(DEST, exist_ok=True)

# Reuse the snapshot that already exists rather than paying to build another.
snaps = requests.get(f"{QDRANT_URL}/collections/{COLLECTION_NAME}/snapshots",
                     timeout=120).json()["result"]
if not snaps:
    r = requests.post(f"{QDRANT_URL}/collections/{COLLECTION_NAME}/snapshots", timeout=3600)
    r.raise_for_status()
    snaps = [r.json()["result"]]

# Sort by NAME, not creation_time — this Qdrant build returns creation_time=None.
# The name embeds the timestamp, so lexicographic order is chronological.
snap = sorted(snaps, key=lambda s: s["name"])[-1]
snap_name = snap["name"]
print(f"snapshot: {snap_name}  ({(snap.get('size') or 0)/1e9:.2f} GB)")

# Pull it through the HTTP API — works regardless of where Qdrant puts it on disk
# (snapshots live under SNAPSHOTS_PATH, NOT under STORAGE_PATH; that was the bug).
out = os.path.join(DEST, snap_name)
url = f"{QDRANT_URL}/collections/{COLLECTION_NAME}/snapshots/{snap_name}"
t0, done, next_mark = time.time(), 0, 512 << 20
with requests.get(url, stream=True, timeout=7200) as r:
    r.raise_for_status()
    with open(out, "wb") as f:
        for chunk in r.iter_content(chunk_size=8 << 20):   # streamed, never held in RAM
            f.write(chunk); done += len(chunk)
            if done >= next_mark:
                print(f"   {done/1e9:.2f} GB  ({(time.time()-t0)/60:.1f} min)")
                next_mark += 512 << 20
print(f"copied {done/1e9:.2f} GB in {(time.time()-t0)/60:.1f} min")

if done < 1_000_000:
    raise RuntimeError(f"snapshot only {done} bytes — download failed, do NOT trust it")

# The query-side contract. Restoring vectors alone is not enough: a wrong prefix
# or threshold silently changes every answer, with no error.
manifest = {
    "collection":     COLLECTION_NAME,
    "snapshot":       snap_name,
    "embed_model":    EMBED_MODEL_NAME,
    "model_tag":      MODEL_TAG,
    "embed_dim":      EMBED_DIM,
    "max_seq_length": MODEL_MAX_TOKENS,
    "query_prefix":   QUERY_PREFIX,
    "doc_prefix":     DOC_PREFIX,
    "distance":       "COSINE (on L2-normalised vectors)",
    "hnsw":           {"m": 16, "ef_construct": 128},
    "n_chunks":       len(unified),
    "kcc_max_chunks": KCC_MAX_CHUNKS,
    "tiers":          {"fallback": TIER_FALLBACK, "grounded": TIER_GROUNDED},
    "fusion_weights": FUSION_WEIGHTS,
    "top_k_default":  TOP_K_DEFAULT,
    "built_utc":      time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
with open(os.path.join(DEST, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

for fn in ("rag_eval_report.json", "retrieval_inspection.json", "shard_progress.json"):
    p = os.path.join(RAG_DB_DIR, fn)
    if os.path.exists(p):
        shutil.copy(p, DEST)

print(f"\nEXPORTED -> {DEST}")
for f in sorted(os.listdir(DEST)):
    print(f"   {os.path.getsize(os.path.join(DEST, f))/1e6:10,.1f} MB  {f}")
print("\nAlso keep EMB_CACHE_DIR — it rebuilds the index without re-running the GPU stage.")

In [ ]:
# ---------------------------------------------------------------------------
# 9. RESTORE — bring the production index back up in a fresh runtime
# ---------------------------------------------------------------------------
# Run this INSTEAD of the ingestion + build cells when you only want to QUERY an
# already-built index. Needs: the server cell above (section 0) to be running,
# and the snapshot + manifest on Drive.
#
# Set RESTORE_MODE = True and run: server cell -> this cell -> tool cell -> eval.
# Skip the PDF/KCC load, the summary, and the build cell entirely.
RESTORE_MODE = False

if RESTORE_MODE:
    import os, json, requests
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    SRC = "/content/drive/MyDrive/rag_production_bge_m3"
    manifest = json.load(open(os.path.join(SRC, "manifest.json"), encoding="utf-8"))
    print(json.dumps(manifest, indent=2)[:700], "...")

    # Restore the collection into the running server from the snapshot file.
    snap = os.path.join(SRC, manifest["snapshot"])
    print(f"\nuploading snapshot ({os.path.getsize(snap)/1e9:.2f} GB) — this is slow ...")
    with open(snap, "rb") as fh:
        r = requests.post(
            f"{QDRANT_URL}/collections/{manifest['collection']}/snapshots/upload?priority=snapshot",
            files={"snapshot": (manifest["snapshot"], fh)}, timeout=7200)
    r.raise_for_status()

    # Re-create the exact query-side contract from the manifest. Getting any of
    # these wrong silently changes every result, which is why they are stored.
    COLLECTION_NAME  = manifest["collection"]
    EMBED_MODEL_NAME = manifest["embed_model"]
    MODEL_MAX_TOKENS = manifest["max_seq_length"]
    QUERY_PREFIX     = manifest["query_prefix"]
    DOC_PREFIX       = manifest["doc_prefix"]
    TIER_FALLBACK    = manifest["tiers"]["fallback"]
    TIER_GROUNDED    = manifest["tiers"]["grounded"]
    FUSION_WEIGHTS   = manifest["fusion_weights"]
    TOP_K_DEFAULT    = manifest["top_k_default"]
    embed_doc   = lambda t: DOC_PREFIX + t
    embed_query = lambda t: QUERY_PREFIX + t

    from qdrant_client import QdrantClient
    from sentence_transformers import SentenceTransformer
    import torch
    qdrant_client = QdrantClient(url=QDRANT_URL, timeout=600)
    embed_model = SentenceTransformer(EMBED_MODEL_NAME,
                                      device="cuda" if torch.cuda.is_available() else "cpu")
    embed_model.max_seq_length = MODEL_MAX_TOKENS

    info = qdrant_client.get_collection(COLLECTION_NAME)
    print(f"\nrestored '{COLLECTION_NAME}': {info.points_count:,} points, "
          f"{info.indexed_vectors_count:,} indexed, status={info.status}")
    print("Now run the TOOL cell (section 4) and query away — skip ingestion/build.")
else:
    print("RESTORE_MODE = False — set it True in a fresh runtime to serve the saved index.")
